In [9]:
from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path(".").resolve()
MAPPINGS_DIR = BASE_DIR / "mappings"


In [10]:

rows = []

for fp in sorted(MAPPINGS_DIR.glob("*.json")):
    firm_id = fp.stem
    data = json.loads(fp.read_text(encoding="utf-8"))

    for v in data.get("variables", []):
        if v.get("needs_manual_review", False):
            final_choice = v.get("final_choice", [])
            # make final_choice readable
            final_str = "; ".join([f"{x['sheet_name']}::{x['row_label']}" for x in final_choice]) if final_choice else ""
            rows.append({
                "firm_id": firm_id,
                "variable": v.get("variable", ""),
                "notes": v.get("notes", ""),
                "final_choice": final_str,
                "num_candidates": len(v.get("candidates", [])),
            })

if len(rows) == 0:
    print("No manual reviews needed. All mappings have been reviewed and finalized.")
else:
    df_review = pd.DataFrame(rows).sort_values(["variable", "firm_id"]).reset_index(drop=True)

    print(f"Manual review needed for {len(df_review)} firm-variable mappings across {df_review['firm_id'].nunique() if len(df_review) else 0} firms.")
    display(df_review)

No manual reviews needed. All mappings have been reviewed and finalized.
